# microWakeWord — Train any wake word

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Two modes:
- **`generate`** — Piper TTS generates ~30k positive samples + your confusables in Colab.
  Easiest path. Requires an IPA pronunciation of your wake word.
- **`bundle`** — You upload a zip with your own samples (real recordings, ElevenLabs
  voices, accent-matched TTS). Higher quality but you do the prep work.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all**, walk away ~45 minutes
3. Find `<output_name>.tflite` + `.json` in your Drive folder when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.


In [7]:
# ╔════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                 ║
# ╚════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Mira"
OUTPUT_NAME = "mira"
AUTHOR = "Ali"
AUTHOR_WEBSITE = ""

# ─── Mode ───
MODE = "generate"

# ─── Drive folder ───
DRIVE_FOLDER = "wakeword_training_mira"

# ─── Positive samples ───
# Mira = "MEE-rah"
WAKE_WORD_IPA_US = "ˈmiːɹə"
WAKE_WORD_IPA_UK = "ˈmiːrə"

SAMPLES_US = 30000
SAMPLES_UK = 15000

# ─── Hard negatives: things that should NOT wake Mira ───
CONFUSABLE_PHRASES = [
    "Mia",
    "Mila",
    "Mina",
    "Myra",
    "Kira",
    "Vera",
    "mirror",
    "Miranda",
    "miracle",
    "nearer",
    "Siri",
    "Alexa",
    "Gemini",
    "hey Google",
    "okay Google",
    "okay Nabu",
]

SAMPLES_PER_CONFUSABLE = 1000

# ─── Bundle mode unused ───
BUNDLE_NAME = "data_bundle.zip"

# ─── Initial wake sensitivity ───
PROBABILITY_CUTOFF = 0.85
SLIDING_WINDOW_SIZE = 5
TENSOR_ARENA_SIZE = 50000

# ─── Manifest metadata ───
TRAINED_LANGUAGES = ["en"]

print(f"Training '{WAKE_WORD}' as {OUTPUT_NAME} in mode={MODE!r}")
print(f"Output → /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite")

Training 'Mira' as mira in mode='generate'
Output → /content/drive/MyDrive/wakeword_training_mira/mira.tflite


In [8]:
# === Mount Drive ===
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')
if MODE == 'bundle':
    BUNDLE_PATH = f'{DRIVE_DIR}/{BUNDLE_NAME}'
    assert os.path.exists(BUNDLE_PATH), (
        f'MODE=bundle but {BUNDLE_PATH} does not exist. Upload your data zip there.')
    print(f'Found bundle: {os.path.getsize(BUNDLE_PATH)/1024/1024:.1f} MB')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive folder: /content/drive/MyDrive/wakeword_training_mira


In [9]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja', 'numpy',
    'pymicro-features', 'pyyaml', 'tensorflow>=2.16', 'webrtcvad-wheels',
    'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/kahrendt/microWakeWord'], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


Installing dependencies...
OK: microwakeword.audio.* imports clean


In [10]:
# === Data preparation ===
# Mode-aware: either unzip user's bundle, or generate samples inline via Piper.
import os, zipfile

os.chdir('/content')

if MODE == 'bundle':
    print(f'Extracting {BUNDLE_PATH}...')
    with zipfile.ZipFile(BUNDLE_PATH, 'r') as zf:
        zf.extractall('/content')
    for d in ['generated_samples', 'real_recordings', 'confusable_negatives']:
        p = f'/content/{d}'
        if os.path.exists(p):
            n = sum(1 for _ in os.scandir(p) if _.name.endswith('.wav'))
            print(f'  {d}: {n} WAVs')
        else:
            print(f'  {d}: MISSING (will train without)')

elif MODE == 'generate':
    # Defer to the Piper sample-gen cells below
    print('MODE=generate — Piper sample-gen cells will produce samples')

else:
    raise ValueError(f'Unknown MODE: {MODE!r}. Use "bundle" or "generate".')


MODE=generate — Piper sample-gen cells will produce samples


In [11]:
# === Piper sample generator install (skipped if MODE=bundle) ===
if MODE == 'generate':
    import glob, os, shutil, subprocess, sys, urllib.request
    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'pip', 'setuptools', 'wheel', 'cython'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'piper-tts', 'piper-sample-generator'], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/rhasspy/piper-sample-generator',
                        PIPER_SAMPLE_GENERATOR_DIR], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)
    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)
    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()
    subprocess.run(f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
                   shell=True, check=True)
    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'
    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
    MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
    os.makedirs('models', exist_ok=True)
    if not os.path.exists(MODEL_PATH):
        print('Downloading libritts_r model (~75 MB)...')
        urllib.request.urlretrieve(
            'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
            MODEL_PATH)
        urllib.request.urlretrieve(
            'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
            MODEL_CONFIG_PATH)
    print('Piper ready')
else:
    print('Skipped (MODE != generate)')


Piper ready


In [16]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

MODEL_PATH = Path(
    "/content/models/en_US-libritts_r-medium.pt"
)

CONFIG_PATH = Path(
    "/content/models/en_US-libritts_r-medium.pt.json"
)

OUTPUT_DIR = Path(
    "/content/mira_piper_test"
)

PIPER_TRAIN_DIR = "/content/piper/src/python"
PIPER_GENERATOR_DIR = "/content/piper-sample-generator"

print("Python:", sys.version)
print("Model exists:", MODEL_PATH.exists(), MODEL_PATH)
print("Config exists:", CONFIG_PATH.exists(), CONFIG_PATH)

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

shutil.rmtree(
    OUTPUT_DIR,
    ignore_errors=True
)

env = os.environ.copy()

existing_pythonpath = env.get(
    "PYTHONPATH",
    ""
)

env["PYTHONPATH"] = ":".join(
    [
        PIPER_TRAIN_DIR,
        PIPER_GENERATOR_DIR,
        existing_pythonpath,
    ]
)

cmd = [
    sys.executable,
    "-m",
    "piper_sample_generator",
    WAKE_WORD_IPA_US,
    "--phoneme-input",
    "--model",
    str(MODEL_PATH),
    "--max-samples",
    "10",
    "--batch-size",
    "10",
    "--noise-scales",
    "0.5",
    "--noise-scale-ws",
    "0.6",
    "--output-dir",
    str(OUTPUT_DIR),
]

print("\nRUNNING MIRA TEST...\n")

result = subprocess.run(
    cmd,
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print(result.stdout)
print("Return code:", result.returncode)

wav_count = (
    len(list(OUTPUT_DIR.glob("*.wav")))
    if OUTPUT_DIR.exists()
    else 0
)

print("TEST WAV FILES:", wav_count)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Model exists: True /content/models/en_US-libritts_r-medium.pt
Config exists: True /content/models/en_US-libritts_r-medium.pt.json
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB

RUNNING MIRA TEST...

DEBUG:__main__:Loading /content/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/1 complete
INFO:__main__:Done

Return code: 0
TEST WAV FILES: 10


In [17]:
# === Generate positive samples (US English, optionally UK) ===
if MODE == 'generate':
    import os
    import subprocess
    import sys

    MODEL_PATH = '/content/models/en_US-libritts_r-medium.pt'
    PIPER_BATCH = 256

    PIPER_TRAIN_DIR = '/content/piper/src/python'
    PIPER_GENERATOR_DIR = '/content/piper-sample-generator'

    env = os.environ.copy()

    existing_pythonpath = env.get('PYTHONPATH', '')

    env['PYTHONPATH'] = ':'.join([
        PIPER_TRAIN_DIR,
        PIPER_GENERATOR_DIR,
        existing_pythonpath,
    ])

    def run_piper(target_word, max_samples, output_dir):
        cmd = [
            sys.executable,
            '-m',
            'piper_sample_generator',
            target_word,
            '--phoneme-input',
            '--model',
            MODEL_PATH,
            '--max-samples',
            str(max_samples),
            '--batch-size',
            str(PIPER_BATCH),
            '--noise-scales',
            '0.5',
            '--noise-scale-ws',
            '0.6',
            '--output-dir',
            output_dir,
        ]

        subprocess.run(
            cmd,
            check=True,
            env=env,
        )

    os.makedirs(
        '/content/generated_samples',
        exist_ok=True,
    )

    print(
        f'Generating {SAMPLES_US} US samples '
        f'for {WAKE_WORD_IPA_US!r}...'
    )

    run_piper(
        WAKE_WORD_IPA_US,
        SAMPLES_US,
        '/content/generated_samples',
    )

    if WAKE_WORD_IPA_UK and SAMPLES_UK > 0:
        print(
            f'Generating {SAMPLES_UK} additional samples '
            f'for {WAKE_WORD_IPA_UK!r}...'
        )

        run_piper(
            WAKE_WORD_IPA_UK,
            SAMPLES_UK,
            '/content/generated_samples',
        )

    n = sum(
        1
        for f in os.listdir('/content/generated_samples')
        if f.endswith('.wav')
    )

    print(f'Total positive samples: {n}')

else:
    print('Skipped (MODE != generate)')

Generating 30000 US samples for 'ˈmiːɹə'...
Generating 15000 additional samples for 'ˈmiːrə'...
Total positive samples: 30000


In [18]:
# === Generate confusable negatives ===
if MODE == 'generate':
    import os
    import shutil
    import subprocess
    import sys
    from pathlib import Path

    MODEL_PATH = '/content/models/en_US-libritts_r-medium.pt'
    PIPER_BATCH = 256

    PIPER_TRAIN_DIR = '/content/piper/src/python'
    PIPER_GENERATOR_DIR = '/content/piper-sample-generator'

    env = os.environ.copy()

    existing_pythonpath = env.get('PYTHONPATH', '')

    env['PYTHONPATH'] = ':'.join([
        PIPER_TRAIN_DIR,
        PIPER_GENERATOR_DIR,
        existing_pythonpath,
    ])

    output_root = Path(
        '/content/confusable_negatives'
    )

    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    total_target = (
        len(CONFUSABLE_PHRASES)
        * SAMPLES_PER_CONFUSABLE
    )

    print(
        f'Generating hard negatives for '
        f'{len(CONFUSABLE_PHRASES)} phrases'
    )

    print(
        f'Target total: {total_target} samples'
    )

    for index, phrase in enumerate(
        CONFUSABLE_PHRASES,
        start=1,
    ):
        safe = ''.join(
            c if c.isalnum() else '_'
            for c in phrase.lower()
        ).strip('_')

        phrase_dir = (
            output_root /
            safe
        )

        existing = (
            len(list(phrase_dir.glob('*.wav')))
            if phrase_dir.exists()
            else 0
        )

        print(
            f'\n[{index}/{len(CONFUSABLE_PHRASES)}] '
            f'{phrase!r}'
        )

        if existing >= SAMPLES_PER_CONFUSABLE:
            print(
                f'Already have {existing}; skipping.'
            )
            continue

        shutil.rmtree(
            phrase_dir,
            ignore_errors=True,
        )

        phrase_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        cmd = [
            sys.executable,
            '-m',
            'piper_sample_generator',
            phrase,
            '--model',
            MODEL_PATH,
            '--max-samples',
            str(SAMPLES_PER_CONFUSABLE),
            '--batch-size',
            str(PIPER_BATCH),
            '--noise-scales',
            '0.5',
            '--noise-scale-ws',
            '0.6',
            '--output-dir',
            str(phrase_dir),
        ]

        subprocess.run(
            cmd,
            check=True,
            env=env,
        )

        produced = len(
            list(
                phrase_dir.glob('*.wav')
            )
        )

        print(
            f'Generated {produced} samples.'
        )

    total = len(
        list(
            output_root.rglob('*.wav')
        )
    )

    print(
        f'\nTotal confusable negatives: {total}'
    )

else:
    print('Skipped (MODE != generate)')

Generating hard negatives for 16 phrases
Target total: 16000 samples

[1/16] 'Mia'
Generated 1000 samples.

[2/16] 'Mila'
Generated 1000 samples.

[3/16] 'Mina'
Generated 1000 samples.

[4/16] 'Myra'
Generated 1000 samples.

[5/16] 'Kira'
Generated 1000 samples.

[6/16] 'Vera'
Generated 1000 samples.

[7/16] 'mirror'
Generated 1000 samples.

[8/16] 'Miranda'
Generated 1000 samples.

[9/16] 'miracle'
Generated 1000 samples.

[10/16] 'nearer'
Generated 1000 samples.

[11/16] 'Siri'
Generated 1000 samples.

[12/16] 'Alexa'
Generated 1000 samples.

[13/16] 'Gemini'
Generated 1000 samples.

[14/16] 'hey Google'
Generated 1000 samples.

[15/16] 'okay Google'
Generated 1000 samples.

[16/16] 'okay Nabu'
Generated 1000 samples.

Total confusable negatives: 16000


In [19]:
from pathlib import Path
import shutil

root = Path("/content/confusable_negatives")

# Move WAVs from phrase subfolders into the root folder
for src in list(root.rglob("*.wav")):
    if src.parent == root:
        continue

    prefix = src.parent.name
    dest = root / f"{prefix}_{src.name}"

    # Extra collision protection
    i = 1
    while dest.exists():
        dest = root / f"{prefix}_{i}_{src.name}"
        i += 1

    shutil.move(str(src), str(dest))

# Remove now-empty subfolders
for folder in sorted(
    [p for p in root.rglob("*") if p.is_dir()],
    reverse=True
):
    try:
        folder.rmdir()
    except OSError:
        pass

positive_count = len(
    list(Path("/content/generated_samples").glob("*.wav"))
)

negative_count = len(
    list(root.glob("*.wav"))
)

print("Positive WAVs:", positive_count)
print("Confusable WAVs:", negative_count)

print(
    "READY:",
    positive_count >= 45000
    and negative_count >= 16000
)

Positive WAVs: 30000
Confusable WAVs: 16000
READY: False


In [20]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

MODEL_PATH = "/content/models/en_US-libritts_r-medium.pt"
TEMP_DIR = Path("/content/mira_missing_us")
FINAL_DIR = Path("/content/generated_samples")

# Start clean
shutil.rmtree(TEMP_DIR, ignore_errors=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# Make piper_train visible to the subprocess
env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/piper/src/python",
    "/content/piper-sample-generator",
    env.get("PYTHONPATH", ""),
])

print("Generating the missing 15,000 US Mira samples...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "piper_sample_generator",
        WAKE_WORD_IPA_US,
        "--phoneme-input",
        "--model",
        MODEL_PATH,
        "--max-samples",
        "15000",
        "--batch-size",
        "256",
        "--noise-scales",
        "0.5",
        "--noise-scale-ws",
        "0.6",
        "--output-dir",
        str(TEMP_DIR),
    ],
    check=True,
    env=env,
)

print("Generation finished. Merging without overwriting...")

for src in TEMP_DIR.glob("*.wav"):
    dest = FINAL_DIR / f"us_extra_{src.name}"
    shutil.move(str(src), str(dest))

shutil.rmtree(TEMP_DIR, ignore_errors=True)

positive_count = len(list(FINAL_DIR.glob("*.wav")))
negative_count = len(
    list(Path("/content/confusable_negatives").glob("*.wav"))
)

print()
print("Positive WAVs:", positive_count)
print("Confusable WAVs:", negative_count)
print(
    "READY:",
    positive_count >= 45000
    and negative_count >= 16000
)

Generating the missing 15,000 US Mira samples...
Generation finished. Merging without overwriting...

Positive WAVs: 45000
Confusable WAVs: 16000
READY: True


In [22]:
# === MINIMAL Mira augmentation + feature extraction ===
# Uses ONLY our 45k Mira positives + 16k confusables.
# No Hugging Face datasets, FMA, AudioSet, or external background corpus.

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

from pathlib import Path
import os
import shutil
import traceback

# ------------------------------------------------------------
# Verify our custom dataset survived
# ------------------------------------------------------------

positive_count = len(
    list(Path("/content/generated_samples").glob("*.wav"))
)

confusable_count = len(
    list(Path("/content/confusable_negatives").glob("*.wav"))
)

print("Mira positives:", positive_count)
print("Confusable negatives:", confusable_count)

assert positive_count >= 45000, (
    f"Expected 45000 Mira positives, found {positive_count}"
)

assert confusable_count >= 16000, (
    f"Expected 16000 confusables, found {confusable_count}"
)

# Remove feature folders left behind by the earlier failed Run All.
# THIS DOES NOT DELETE THE WAV FILES.
shutil.rmtree(
    "/content/generated_augmented_features",
    ignore_errors=True
)

shutil.rmtree(
    "/content/confusable_features",
    ignore_errors=True
)

# ------------------------------------------------------------
# Lightweight augmentation
# ------------------------------------------------------------

augmenter = Augmentation(
    augmentation_duration_s=3.2,

    augmentation_probabilities={
        "SevenBandParametricEQ": 0.15,
        "TanhDistortion": 0.10,
        "PitchShift": 0.15,
        "BandStopFilter": 0.10,

        # Generated noise only — no downloaded audio corpus.
        "AddColorNoise": 0.30,

        # Disabled for this first-pass model.
        "AddBackgroundNoise": 0.0,
        "RIR": 0.0,

        "Gain": 1.00,
        "GainTransition": 0.25,
    },

    # Empty paths explicitly disable these external augmentations.
    impulse_paths=[],
    background_paths=[],

    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

# ------------------------------------------------------------
# Splits
# ------------------------------------------------------------

SPLIT_CONFIG = {
    "training": {
        "split_name": "train",
        "repetition": 3,
        "slide_frames": 10,
    },
    "validation": {
        "split_name": "validation",
        "repetition": 1,
        "slide_frames": 10,
    },
    "testing": {
        "split_name": "test",
        "repetition": 1,
        "slide_frames": 1,
    },
}

# ------------------------------------------------------------
# Mira POSITIVE features
# ------------------------------------------------------------

print("\n=== MIRA POSITIVE FEATURES ===")

positive_clips = Clips(
    input_directory="/content/generated_samples",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)

for split, cfg in SPLIT_CONFIG.items():

    out = f"/content/generated_augmented_features/{split}"
    mmap = f"{out}/wakeword_mmap"

    os.makedirs(out, exist_ok=True)

    print(
        f"\nGenerating Mira {split} "
        f"(repeat={cfg['repetition']})..."
    )

    try:
        sg = SpectrogramGeneration(
            clips=positive_clips,
            augmenter=augmenter,
            slide_frames=cfg["slide_frames"],
            step_ms=10,
        )

        RaggedMmap.from_generator(
            out_dir=mmap,
            batch_size=200,
            verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg["split_name"],
                repeat=cfg["repetition"],
            ),
        )

    except Exception:
        traceback.print_exc()

        if os.path.exists(mmap):
            shutil.rmtree(
                mmap,
                ignore_errors=True
            )

        raise

print("\nMira positive features ready.")

# ------------------------------------------------------------
# CONFUSABLE NEGATIVE features
# ------------------------------------------------------------

print("\n=== CONFUSABLE NEGATIVE FEATURES ===")

confusable_clips = Clips(
    input_directory="/content/confusable_negatives",
    file_pattern="*.wav",
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)

for split, cfg in SPLIT_CONFIG.items():

    out = f"/content/confusable_features/{split}"
    mmap = f"{out}/wakeword_mmap"

    os.makedirs(out, exist_ok=True)

    print(
        f"\nGenerating confusable {split} "
        f"(repeat={cfg['repetition']})..."
    )

    try:
        sg = SpectrogramGeneration(
            clips=confusable_clips,
            augmenter=augmenter,
            slide_frames=cfg["slide_frames"],
            step_ms=10,
        )

        RaggedMmap.from_generator(
            out_dir=mmap,
            batch_size=200,
            verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg["split_name"],
                repeat=cfg["repetition"],
            ),
        )

    except Exception:
        traceback.print_exc()

        if os.path.exists(mmap):
            shutil.rmtree(
                mmap,
                ignore_errors=True
            )

        raise

print("\nConfusable features ready.")

print("\n================================")
print("MIRA FEATURE EXTRACTION COMPLETE")
print("================================")

Mira positives: 45000
Confusable negatives: 16000

=== MIRA POSITIVE FEATURES ===

Generating Mira training (repeat=3)...


0it [00:00, ?it/s]


Generating Mira validation (repeat=1)...


0it [00:00, ?it/s]


Generating Mira testing (repeat=1)...


...


Generating confusable validation (repeat=1)...


0it [00:00, ?it/s]


Generating confusable testing (repeat=1)...


0it [00:00, ?it/s]


Confusable features ready.

MIRA FEATURE EXTRACTION COMPLETE


In [ ]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


In [23]:
# === Mira-only Training config YAML ===

import yaml
import os
from pathlib import Path

positive_mmap = Path(
    "/content/generated_augmented_features/training/wakeword_mmap"
)

confusable_mmap = Path(
    "/content/confusable_features/training/wakeword_mmap"
)

assert positive_mmap.exists(), (
    f"Missing positive features: {positive_mmap}"
)

assert confusable_mmap.exists(), (
    f"Missing confusable features: {confusable_mmap}"
)

config = {
    "window_step_ms": 10,

    "train_dir": f"trained_models/{OUTPUT_NAME}",

    "features": [
        {
            "features_dir": "generated_augmented_features",
            "sampling_weight": 8.0,
            "penalty_weight": 2.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },

        {
            "features_dir": "confusable_features",
            "sampling_weight": 8.0,
            "penalty_weight": 5.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
    ],

    "training_steps": [
        25000,
        20000,
    ],

    "positive_class_weight": [
        2,
        2,
    ],

    "negative_class_weight": [
        40,
        50,
    ],

    "learning_rates": [
        0.001,
        0.0001,
    ],

    "batch_size": 256,

    "time_mask_max_size": [5, 5],
    "time_mask_count": [1, 1],

    "freq_mask_max_size": [3, 3],
    "freq_mask_count": [1, 1],

    "eval_step_interval": 500,

    "clip_duration_ms": 1500,

    "target_minimization": 0.4,

    "minimization_metric": "ambient_false_positives_per_hour",

    "maximization_metric": "average_viable_recall",
}

os.makedirs(
    f"trained_models/{OUTPUT_NAME}",
    exist_ok=True,
)

with open(
    "/content/training_parameters.yaml",
    "w",
) as f:
    yaml.dump(
        config,
        f,
        sort_keys=False,
    )

print("training_parameters.yaml ready")
print("Feature sets:", len(config["features"]))
print("Total training steps:", sum(config["training_steps"]))
print()
print(
    open(
        "/content/training_parameters.yaml"
    ).read()
)

training_parameters.yaml ready
Feature sets: 2
Total training steps: 45000

window_step_ms: 10
train_dir: trained_models/mira
features:
- features_dir: generated_augmented_features
  sampling_weight: 8.0
  penalty_weight: 2.0
  truth: true
  truncation_strategy: truncate_start
  type: mmap
- features_dir: confusable_features
  sampling_weight: 8.0
  penalty_weight: 5.0
  truth: false
  truncation_strategy: random
  type: mmap
training_steps:
- 25000
- 20000
positive_class_weight:
- 2
- 2
negative_class_weight:
- 40
- 50
learning_rates:
- 0.001
- 0.0001
batch_size: 256
time_mask_max_size:
- 5
- 5
time_mask_count:
- 1
- 1
freq_mask_max_size:
- 3
- 3
freq_mask_count:
- 1
- 1
eval_step_interval: 500
clip_duration_ms: 1500
target_minimization: 0.4
minimization_metric: ambient_false_positives_per_hour
maximization_metric: average_viable_recall



In [24]:
import yaml

path = "/content/training_parameters.yaml"

with open(path, "r") as f:
    config = yaml.safe_load(f)

# For our first Mira-only model, choose the best checkpoint
# using the validation loss + recall that actually exist.
config["minimization_metric"] = "loss"
config["target_minimization"] = 0.40
config["maximization_metric"] = "recall"

with open(path, "w") as f:
    yaml.dump(config, f, sort_keys=False)

print("MIRA TRAINING CONFIG PATCHED")
print("Minimize:", config["minimization_metric"])
print("Target:", config["target_minimization"])
print("Maximize:", config["maximization_metric"])

MIRA TRAINING CONFIG PATCHED
Minimize: loss
Target: 0.4
Maximize: recall


In [27]:
import shutil
from pathlib import Path

train_dir = Path("/content/trained_models/mira")

print("Exists before:", train_dir.exists())

shutil.rmtree(
    train_dir,
    ignore_errors=True
)

print("Exists after:", train_dir.exists())

Exists before: True
Exists after: False


In [28]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

TRAIN_DIR = Path("/content/trained_models/mira")

print("Removing stale training folder...")
shutil.rmtree(TRAIN_DIR, ignore_errors=True)

print("Training folder exists after cleanup:", TRAIN_DIR.exists())

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/microWakeWord",
    env.get("PYTHONPATH", ""),
])

cmd = [
    sys.executable,
    "-m",
    "microwakeword.model_train_eval",
    "--training_config",
    "/content/training_parameters.yaml",
    "--train",
    "1",
    "--restore_checkpoint",
    "0",
    "--test_tf_nonstreaming",
    "0",
    "--test_tflite_nonstreaming",
    "0",
    "--test_tflite_streaming",
    "0",
    "--test_tflite_streaming_quantized",
    "0",
    "inception",
    "--cnn1_filters",
    "32",
    "--cnn1_kernel_sizes",
    "5",
    "--cnn1_subspectral_groups",
    "4",
    "--cnn2_filters1",
    "24,24,24",
    "--cnn2_filters2",
    "32,64,96",
    "--cnn2_kernel_sizes",
    "3,5,5",
    "--cnn2_subspectral_groups",
    "1,1,1",
    "--cnn2_dilation",
    "1,1,1",
    "--dropout",
    "0.8",
]

print("\nSTARTING CLEAN MIRA TRAINING...\n")

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()

print("\nExit code:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError("Training failed — see output above.")


Streaming output truncated to the last 5000 lines.
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #34
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #35
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #36
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0007; Mini-Batch #37
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0007; Mini-Batch #38
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #39
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #40
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #41
Validation Batch #81: Accuracy = 1.000; Recall = 1.000; Precision = 1.000; Loss = 0.0006; Mini-Batch #42
Vali

In [29]:
from pathlib import Path

root = Path("/content/trained_models/mira")

print("TRAINING OUTPUTS:")

for p in sorted(root.rglob("*")):
    if p.is_file():
        print(p)

TRAINING OUTPUTS:
/content/trained_models/mira/best_weights.weights.h5
/content/trained_models/mira/last_weights.weights.h5
/content/trained_models/mira/logs/train/events.out.tfevents.1788522012.93f8b28a76f3.39762.0.v2
/content/trained_models/mira/logs/validation/events.out.tfevents.1788522012.93f8b28a76f3.39762.1.v2
/content/trained_models/mira/model_summary.txt
/content/trained_models/mira/restore/checkpoint
/content/trained_models/mira/restore/ckpt-1.data-00000-of-00001
/content/trained_models/mira/restore/ckpt-1.index
/content/trained_models/mira/restore/ckpt-2.data-00000-of-00001
/content/trained_models/mira/restore/ckpt-2.index
/content/trained_models/mira/restore/ckpt-3.data-00000-of-00001
/content/trained_models/mira/restore/ckpt-3.index
/content/trained_models/mira/restore/ckpt-4.data-00000-of-00001
/content/trained_models/mira/restore/ckpt-4.index
/content/trained_models/mira/restore/ckpt-5.data-00000-of-00001
/content/trained_models/mira/restore/ckpt-5.index
/content/trained

In [30]:
import os
import sys
import subprocess

env = os.environ.copy()

env["PYTHONPATH"] = ":".join([
    "/content/microWakeWord",
    env.get("PYTHONPATH", ""),
])

cmd = [
    sys.executable,
    "-m",
    "microwakeword.model_train_eval",

    "--training_config",
    "/content/training_parameters.yaml",

    # DO NOT TRAIN AGAIN
    "--train",
    "0",

    "--restore_checkpoint",
    "0",

    # Use the best weights from the completed training run
    "--use_weights",
    "best_weights",

    # We only want the quantized streaming model
    "--test_tf_nonstreaming",
    "0",

    "--test_tflite_nonstreaming",
    "0",

    "--test_tflite_nonstreaming_quantized",
    "0",

    "--test_tflite_streaming",
    "0",

    "--test_tflite_streaming_quantized",
    "1",

    # Same architecture used for training
    "inception",

    "--cnn1_filters",
    "32",

    "--cnn1_kernel_sizes",
    "5",

    "--cnn1_subspectral_groups",
    "4",

    "--cnn2_filters1",
    "24,24,24",

    "--cnn2_filters2",
    "32,64,96",

    "--cnn2_kernel_sizes",
    "3,5,5",

    "--cnn2_subspectral_groups",
    "1,1,1",

    "--cnn2_dilation",
    "1,1,1",

    "--dropout",
    "0.8",
]

print("EXPORTING MIRA QUANTIZED STREAMING MODEL...\n")

proc = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()

print("\nExit code:", proc.returncode)

if proc.returncode != 0:
    raise RuntimeError(
        "Mira TFLite export failed — see output above."
    )

EXPORTING MIRA QUANTIZED STREAMING MODEL...

    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
INFO:absl:Loading and analyzing data sets.
2026-09-04 12:15:30.313417: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1788524130.314439   51776 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:00:05.0, compute capability: 8.0
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━

In [31]:
from pathlib import Path

tflite = Path(
    "/content/trained_models/mira/"
    "tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)

print("MIRA TFLITE EXISTS:", tflite.exists())

if tflite.exists():
    print("SIZE:", round(tflite.stat().st_size / 1024, 1), "KB")
    print("PATH:", tflite)

MIRA TFLITE EXISTS: True
SIZE: 124.1 KB
PATH: /content/trained_models/mira/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite


In [32]:
# === Final Mira export + push to Drive ===

import os
import json
import shutil
import datetime

tflite_src = (
    "/content/trained_models/mira/"
    "tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)

assert os.path.exists(tflite_src), (
    f"No model at {tflite_src}"
)

OUT_TFLITE = "/content/mira.tflite"
OUT_JSON = "/content/mira.json"

# Copy trained model
shutil.copy2(
    tflite_src,
    OUT_TFLITE
)

print(
    "wrote mira.tflite",
    f"({os.path.getsize(OUT_TFLITE) / 1024:.1f} KB)"
)

# Manifest for our Android runtime
manifest = {
    "type": "micro",
    "wake_word": "Mira",
    "author": "Ali",
    "website": "",
    "model": "mira.tflite",
    "trained_languages": ["en"],
    "version": 2,
    "micro": {
        "probability_cutoff": 0.85,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 50000,
        "minimum_esphome_version": "2024.7.0"
    }
}

with open(
    OUT_JSON,
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

print("wrote mira.json")

# Push to Drive
DRIVE_OUT = (
    f"/content/drive/MyDrive/"
    f"{DRIVE_FOLDER}"
)

os.makedirs(
    DRIVE_OUT,
    exist_ok=True
)

for src in [
    OUT_TFLITE,
    OUT_JSON
]:
    dst = os.path.join(
        DRIVE_OUT,
        os.path.basename(src)
    )

    shutil.copy2(
        src,
        dst
    )

    print(
        "pushed",
        os.path.basename(src),
        "->",
        dst
    )

timestamp = datetime.datetime.now(
    datetime.timezone.utc
).isoformat()

with open(
    os.path.join(
        DRIVE_OUT,
        "_run_finished.txt"
    ),
    "w"
) as f:
    f.write(
        f"Mira model export finished at {timestamp}\n"
    )

print()
print("MIRA EXPORT COMPLETE")
print(
    f"Drive folder: "
    f"/content/drive/MyDrive/{DRIVE_FOLDER}"
)

wrote mira.tflite (124.1 KB)
wrote mira.json
pushed mira.tflite -> /content/drive/MyDrive/wakeword_training_mira/mira.tflite
pushed mira.json -> /content/drive/MyDrive/wakeword_training_mira/mira.json

MIRA EXPORT COMPLETE
Drive folder: /content/drive/MyDrive/wakeword_training_mira


## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
